In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import os

# Ensure the models directory exists
os.makedirs('models', exist_ok=True)

# 1. Data Loading & Cleaning
print("Loading text and metadata...")
cols = ['book_id', 'title', 'description', 'num_pages', 'average_rating', 'publication_year']

# Assuming your notebook is in the root directory next to the 'data' folder
books_df = pd.read_csv('../goodreads_books.csv', usecols=cols)

# Drop books without a plot description
books_df = books_df.dropna(subset=['description']).reset_index(drop=True)

# LAPTOP-FRIENDLY FIX: Keep only 50,000 books for the web app demo to prevent freezing
books_df = books_df.head(5000)                 #reduced to see resuls on frontend side
print(f"Dataset reduced to {len(books_df)} books for memory efficiency.")

# Fill missing physical metadata
books_df['num_pages'] = pd.to_numeric(books_df['num_pages'], errors='coerce').fillna(books_df['num_pages'].median())
books_df['average_rating'] = pd.to_numeric(books_df['average_rating'], errors='coerce').fillna(books_df['average_rating'].mean())
books_df['publication_year'] = pd.to_numeric(books_df['publication_year'], errors='coerce').fillna(books_df['publication_year'].median())

# 2. Vectorization & Scaling
print("Vectorizing plot descriptions (10k word limit)...")
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(books_df['description'])

print("Scaling metadata...")
scaler = MinMaxScaler()
meta_features = scaler.fit_transform(books_df[['num_pages', 'average_rating', 'publication_year']])

# 3. Hybrid Matrix Construction
print("Constructing hybrid matrix...")
# Multiply TF-IDF by 3 to ensure plot similarity outweighs physical page count/year
hybrid_matrix = sp.hstack([tfidf_matrix * 3, meta_features], format='csr')
print(f"Final Matrix Shape: {hybrid_matrix.shape}")

# 4. Engine Testing Function
def explore_similar_plots(target_book_id, matrix, df, top_n=20):
    idx_list = df.index[df['book_id'] == target_book_id].tolist()
    if not idx_list:
        return "Target book not found."
    
    idx = idx_list[0]
    sim_scores = cosine_similarity(matrix[idx], matrix).flatten()
    
    # Sort and grab highest scores (excluding the book matching itself)
    top_indices = sim_scores.argsort()[-(top_n+1):-1][::-1]
    return df.iloc[top_indices][['book_id', 'title', 'average_rating', 'num_pages']]

# Test the engine
test_id = books_df['book_id'].iloc[0]
print(f"\n--- Discovering books similar to: {books_df['title'].iloc[0]} ---")
display(explore_similar_plots(test_id, hybrid_matrix, books_df))

# 5. Export Production Assets
print("\nExporting production assets...")
sp.save_npz('models/hybrid_discovery_matrix.npz', hybrid_matrix)
books_df[['book_id', 'title', 'average_rating', 'num_pages']].to_pickle('models/discovery_books_mapping.pkl')
print("Engine exported successfully to the models/ folder.")

Loading text and metadata...
Dataset reduced to 5000 books for memory efficiency.
Vectorizing plot descriptions (10k word limit)...
Scaling metadata...
Constructing hybrid matrix...
Final Matrix Shape: (5000, 10003)

--- Discovering books similar to: Good Harbor ---


,book_id,title,average_rating,num_pages
2462,25740767,KID ROCK,3.00,33.0
1430,140468,L'Homme dans le labyrinthe,3.80,308.0
845,120626,Des Souris et des hommes,3.84,192.0
79,23566096,Lugas,3.91,152.0
2485,35961771,बातमीमागची बातमी [Batamimagachi Batami],4.00,220.0
3565,22314050,My Side of the Mountain,4.03,251.0
3761,6474486,Ensaio Sobre a Cegueira,4.08,310.0
265,16111978,หน้ากากดอกซ่อนกลิ่น,4.06,508.0
4231,9576838,I tre moschettieri,4.06,636.0
4646,25628687,絶頂集 [Zecchoushuu],4.50,108.0



Exporting production assets...
Engine exported successfully to the models/ folder.


In [7]:
sp.save_npz(r'C:\Users\emrid\Book-Recommendation-System-main (1)\bookwise\models\hybrid_discovery_matrix.npz', hybrid_matrix)
books_df[['book_id', 'title', 'average_rating', 'num_pages']].to_pickle(r'C:\Users\emrid\Book-Recommendation-System-main (1)\bookwise\models\discovery_books_mapping.pkl')